In [ ]:
import sys, os
sys.path.insert(0, '../../utils')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from matplotlib.ticker import PercentFormatter
from utils import load_neurons_table, load_synapses_position_transformed
from connectome_types import CONNECTOME_SYN_TABLE_PATH, CONNECTOME_PRE_SYN_TABLE_PATH, SPINE_TABLE_OUTGOING
from neuron_custom_features import calc_spines_features
from spines_utils import filter_valid_neuron_w_spines
from plot_utils import ex_color
from spine_pref_utils import per_neuron_spine_ratio
from stats_corr import add_log_curve
from axon_pref_utils import idan_fit, fit_pop, fit_single_wrap

In [ ]:
neurons_df = load_neurons_table()
syn_df = load_synapses_position_transformed(base_syn_table_path=CONNECTOME_SYN_TABLE_PATH)
df, syn_with_tags = calc_spines_features(neurons_df, syn_df)
df, filtered_syn_mat, filtered_bin_mat, filtered_mapping, filtered_reverse_mapping, ex_neurons, inh_neurons = filter_valid_neuron_w_spines(df)

spine_df_outgoing = pd.read_csv(SPINE_TABLE_OUTGOING)
outgoing_syn_df = load_synapses_position_transformed(base_syn_table_path=CONNECTOME_PRE_SYN_TABLE_PATH)
outgoing_syn_with_tags = outgoing_syn_df[outgoing_syn_df.id_.isin(spine_df_outgoing.target_id)].copy()
outgoing_syn_with_tags['tag'] = outgoing_syn_with_tags.id_.map(spine_df_outgoing.set_index('target_id').tag)

ex_outgoing_syn_with_tags = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_clf_type == 'E']
ex_neurons = ex_neurons.copy()

In [ ]:
main_feature = 'spine'

e_all_outside_stats = per_neuron_spine_ratio(spine_df_outgoing, ex_neurons.root_id.tolist(), group_by='pre_pt_root_id')
ex_neurons['x_all_outside'] = ex_neurons.root_id.map(e_all_outside_stats['n_syn'])
ex_neurons[f'outgoing_{main_feature}_ratio_all_outside'] = ex_neurons.root_id.map(e_all_outside_stats['ratio'])

# Predict spine ratio from fitted model parameters
rho_inf_opt = 0.8212
d0_opt = 124.1607
outgoing_syn = ex_outgoing_syn_with_tags
high_node_id = 864691135617152361
low_node_id = 864691136177632518

predict_values = []
for root_id in tqdm(ex_neurons['root_id']):
    neuron_outgoing_syn = outgoing_syn[outgoing_syn['pre_id'] == root_id]
    distances = neuron_outgoing_syn['dist_to_pre_syn_soma'].values
    if len(distances) == 0:
        predict_values.append(np.nan)
        continue
    predicted_fractions = idan_fit(distances, rho_inf_opt, d0_opt)
    predicted_spine_ratio = predicted_fractions.mean()
    predict_values.append(predicted_spine_ratio)

ex_neurons['predicted_spine_ratio'] = predict_values

x_col = 'x_all_outside'
y_col = f'outgoing_{main_feature}_ratio_all_outside'
y_col_predicted = 'predicted_spine_ratio'

In [ ]:
plt.rcParams['font.size'] = 13
plt.rcParams['font.family'] = 'Arial'

spiny_color  = '#7C3AED'
aspiny_color = '#059669'
scatter_overlay_fontsize = 11

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6*2, 5), dpi=300, sharey=True)

ax = axes[0]
ax.scatter(x=ex_neurons[x_col], y=ex_neurons[y_col], edgecolors=ex_color, facecolors='none', s=5, alpha=0.7)
ax.set_xlabel('Number of output synapses')
add_log_curve(ex_neurons, x_col=x_col, y_col=y_col, ax=ax, color='gray',
              fontsize=scatter_overlay_fontsize, linestyle=':')

high_deg_neuron = ex_neurons[ex_neurons['root_id'] == high_node_id]
low_deg_neuron = ex_neurons[ex_neurons['root_id'] == low_node_id]
ax.scatter(x=high_deg_neuron[x_col], y=high_deg_neuron[y_col],
           color=spiny_color, s=35, zorder=10, label='Neuron in Fig. 3A')
ax.scatter(x=low_deg_neuron[x_col], y=low_deg_neuron[y_col],
           color=aspiny_color, s=35, zorder=10, label='Neuron in Fig. 3B')
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))
ax.set_ylabel('% of output synapses\n on target spines')

ax = axes[1]
ax.scatter(x=ex_neurons[x_col], y=ex_neurons[y_col_predicted], edgecolors=ex_color, facecolors='none', s=5, alpha=0.7)
ax.set_xlabel('Number of output synapses')
ax.set_ylabel('Predicted % of output synapses\n on target spines')
add_log_curve(ex_neurons, x_col=x_col, y_col=y_col_predicted, ax=ax, color='gray',
              fontsize=scatter_overlay_fontsize, linestyle=':')

ax.scatter(x=high_deg_neuron[x_col], y=high_deg_neuron[y_col_predicted],
           color=spiny_color, s=35, zorder=10, label='Neuron in Fig. 3A')
ax.scatter(x=low_deg_neuron[x_col], y=low_deg_neuron[y_col_predicted],
           color=aspiny_color, s=35, zorder=10, label='Neuron in Fig. 3B')
ax.legend(frameon=False, loc='upper right')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.savefig('fig_s8.pdf', format='pdf', bbox_inches='tight')